# Leakage along a subgroup chain

Companion notebook for Fig. 3b. Explore the cached structures or scores used in these panels; this notebook does not reproduce the final figure layout.


## What this demonstrates

Along a maximal subgroup chain, leakage conditioned on a target stays at the numerical floor while that target is retained and turns on when the chain descends past it. The inversion target Ci remains a negative control throughout.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "demo" / "data" / "manifest.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the SymFP repository or its demo directory.")
CODE_ROOT = PROJECT_ROOT / "code" if (PROJECT_ROOT / "code" / "utils" / "demo_utils.py").is_file() else PROJECT_ROOT
sys.path.insert(0, str(CODE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

from utils.demo_utils import (
    ANGULAR_MODES,
    DEGREE_STYLES,
    PALETTE,
    RANK_ACTIVATION_CASES,
    SUBGROUP_BOUNDARIES,
    SUBGROUP_COLUMNS,
    SUBGROUP_ENTERED,
    SUBGROUP_EXPECTED_ONSETS,
    SUBGROUP_TARGETS,
    environment_versions,
    load_angular_mode,
    load_rank_activation,
    load_subgroup_chain,
    runtime_summary,
)

runtime = runtime_summary()
display(environment_versions().style.hide(axis="index"))
print(f"Project root: {runtime['project_root']}")
print(f"Device available to the project: {runtime['device_available']}")

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#555555",
    "axes.linewidth": 0.8,
    "figure.dpi": 120,
})

## Configuration

Select one target to highlight. All target curves remain visible for context.

In [ ]:
TARGET = "D4h"
ONSET_THRESHOLD = 1e-6
assert TARGET in SUBGROUP_TARGETS
print(f"Highlighted target: {TARGET}")
print(f"Operational onset threshold: {ONSET_THRESHOLD:g}")

## Load data

The cached leakage table contains 61 frames: one Oh parent followed by four 15-frame deformation segments. The exact point group changes at frames 1, 16, 31 and 46.

In [ ]:
leakage = load_subgroup_chain()
COLOURS = {
    "Oh": PALETTE["blue"], "D4h": PALETTE["teal"], "D2h": PALETTE["purple"],
    "C2h": PALETTE["coral"], "Ci": PALETTE["deep_purple"],
}
print(f"Loaded {len(leakage)} frames and {len(SUBGROUP_COLUMNS)} target-conditioned leakage curves.")

## Plot

Dashed vertical lines mark the frame at which the structure enters the next subgroup. The selected target is emphasised; other targets remain visible at lower opacity.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for target, column in SUBGROUP_COLUMNS.items():
    selected = target == TARGET
    ax.plot(leakage["frame"], np.maximum(leakage[column], 1e-12),
            color=COLOURS[target], lw=2.5 if selected else 1.4,
            alpha=1.0 if selected else 0.45, label=target, zorder=4 if selected else 2)

for frame, group in zip(SUBGROUP_BOUNDARIES, SUBGROUP_ENTERED):
    ax.axvline(frame, color=PALETTE["grey"], lw=0.9, ls="--", zorder=1)
    ax.text(frame + 0.7, 3e-4, f"enters {group}", rotation=90,
            va="bottom", ha="left", color="#555555", fontsize=8)

ax.set_yscale("log")
ax.set_xlabel("descent along the subgroup chain")
ax.set_ylabel("leakage energy to target group")
ax.set_xticks([])
ax.legend(frameon=False, ncol=5, loc="upper center", bbox_to_anchor=(0.5, 1.14))
ax.set_title("Oh ⊃ D4h ⊃ D2h ⊃ C2h ⊃ Ci")
plt.show()

In [ ]:
onset_rows = []
for target, column in SUBGROUP_COLUMNS.items():
    active = leakage.loc[leakage[column] > ONSET_THRESHOLD, "frame"]
    observed = int(active.iloc[0]) if len(active) else None
    expected = SUBGROUP_EXPECTED_ONSETS[target]
    onset_rows.append({
        "target": target,
        "expected onset frame": expected if expected is not None else "never",
        "observed onset frame": observed if observed is not None else "never",
        "match": observed == expected,
    })
onset_table = pd.DataFrame(onset_rows)
display(onset_table.style.hide(axis="index"))

## Short conclusion

In [ ]:
if onset_table["match"].all():
    print("PASS — the leakage onsets reproduce the subgroup order, and leakage to Ci stays below the onset threshold throughout.")
else:
    print("CHECK — at least one observed onset differs from the expected subgroup-chain boundary.")